# 18 — Barrido de campos: ¿hay información que no estamos usando?

Dos personas del leaderboard llegan a ~0,945 y nosotros a 0,937, así que puede haber un campo con señal sin aprovechar. Plan:
1. **Inventario** de todos los campos del jsonlines crudo (incluidas las sub-claves de dicts y de los elementos de listas): tipo, % nulos/vacíos, valores distintos, y si ya se usa y cómo.
2. Para cada campo **no usado** (o usado solo como conteo/flag): una representación simple y dos mediciones:
   - **(a) señal sola**: accuracy de un HistGB con solo esa feature (con `TargetEncoder` si es categórica), CV de 5 folds;
   - **(b) aporte marginal**: sumada al HistGB P7a (0,9371), con comparación pareada.
3. Tabla ordenada por aporte marginal; marcar señal sola > 0,70.
4. Lo que pase: confirmar en P7c y con `random_state=7`.

**No es leakage:** las categóricas van con `TargetEncoder` (*cross-fitting*) adentro del `Pipeline`; numéricas, flags y conteos son determinísticos y no usan el target. Material: target encoder y *feature engineering* (`AP_Categóricas.ipynb`), Gradient Boosting (`Árboles.pdf`).

In [1]:
import sys
sys.path.append("..")
import os, time, warnings, ast
import numpy as np
import pandas as pd
from joblib import Memory
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate

from features import (aplanar_anidadas, agregar_features_contenido, agregar_combos, armar_pipeline_histgb_combos,
                      PrediccionOOF, armar_modelo_texto)

SEED = 42
crudo = pd.read_json("../data/train_data.jsonlines", lines=True)
y = (crudo["condition"] == "new").astype(int)
train = agregar_combos(agregar_features_contenido(aplanar_anidadas(crudo)))
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
carpeta_lab = ".cache_laboratorio"
memoria_texto = Memory(location=carpeta_lab, verbose=0)
archivo_resultados = os.path.join(carpeta_lab, "resultados.csv")
columnas_folds = ["fold_1", "fold_2", "fold_3", "fold_4", "fold_5"]
NOMBRE_P7A = "P7a - HistGB + TargetEncoder(seller, categoría, 19 combos) + vendedor sin target"
acc_p7a = pd.read_csv(archivo_resultados).set_index("nombre").loc[NOMBRE_P7A, columnas_folds].to_numpy(dtype=float)

## 1. Inventario de campos

In [2]:
def valores_campo(serie, ruta):
    """Valores de una sub-clave (ruta como lista) para cada fila; listas de dicts -> lista de valores."""
    def extraer(v, partes):
        if not partes:
            return v
        if isinstance(v, dict):
            return extraer(v.get(partes[0]), partes[1:])
        if isinstance(v, list):
            return [extraer(e, partes) for e in v]
        return None
    return serie.map(lambda v: extraer(v, ruta))


def resumen(valores):
    def vacio(v):
        return v is None or (isinstance(v, float) and np.isnan(v)) or (isinstance(v, (list, str, dict)) and len(v) == 0)
    def a_texto(v):
        return "|".join(map(str, v)) if isinstance(v, list) else str(v)
    muestra = next((v for v in valores if not vacio(v)), None)
    return {"tipo": type(muestra).__name__ if muestra is not None else "—",
            "pct_nulo_o_vacio": round(valores.map(vacio).mean(), 4),
            "valores_distintos": valores.map(a_texto).nunique()}


uso = {
    "id": "no (identificador)", "site_id": "no (constante)", "title": "sí: TF-IDF 500 + modelo de texto (prob_texto_new)",
    "subtitle": "no (vacío)", "seller_id": "sí: CatBoost crudo; TargetEncoder + conteo/precio (HistGB)",
    "price": "sí: log_price (capeado); bin de precio en combos", "base_price": "solo capeado, no entra al modelo",
    "currency_id": "sí: grupo del capeo; c_currency", "initial_quantity": "sí", "available_quantity": "sí (log)",
    "sold_quantity": "sí", "buying_mode": "sí (categórica)", "listing_type_id": "sí (categórica)",
    "start_time": "sí: hora_inicio, gap_actualizacion", "stop_time": "no", "date_created": "no",
    "last_updated": "sí: gap_actualizacion", "condition": "TARGET", "warranty": "sí: flag + texto en el modelo de texto",
    "accepts_mercadopago": "sí", "category_id": "sí: CatBoost crudo; TargetEncoder (HistGB)",
    "official_store_id": "flag es_tienda_oficial + c_store", "original_price": "solo flag tiene_descuento",
    "video_id": "solo flag tiene_video", "automatic_relist": "sí", "status": "c_status (combos)",
    "parent_item_id": "solo flag tiene_parent_item", "permalink": "no", "thumbnail": "probado en P8 (no suma)",
    "secure_thumbnail": "no (igual a thumbnail)", "listing_source": "no", "international_delivery_mode": "no",
    "differential_pricing": "no", "catalog_product_id": "no", "coverage_areas": "no", "descriptions": "no",
    "tags": "c_tags (combos); flags tag_* no entran", "deal_ids": "solo flag tiene_oferta",
    "sub_status": "no (flag tiene_sub_status no entra)", "seller_address.state.name": "sí: seller_state",
    "seller_address.state.id": "no (equivale a state.name)", "seller_address.city.name": "no",
    "seller_address.city.id": "no", "seller_address.country.name": "no", "seller_address.country.id": "no",
    "shipping.mode": "sí: shipping_mode", "shipping.free_shipping": "sí", "shipping.local_pick_up": "sí",
    "shipping.methods": "no", "shipping.tags": "no", "shipping.dimensions": "no", "shipping.free_methods": "no",
    "non_mercado_pago_payment_methods[].description": "conteo n_metodos_pago + c_pagos",
    "non_mercado_pago_payment_methods[].id": "no (equivale a description)", "non_mercado_pago_payment_methods[].type": "no",
    "variations[].attribute_combinations": "solo conteo n_variaciones", "variations[].seller_custom_field": "no",
    "variations[].picture_ids": "no", "variations[].sold_quantity": "no", "variations[].available_quantity": "no",
    "variations[].id": "no", "variations[].price": "no",
    "attributes[].id": "conteo n_atributos + c_attr", "attributes[].name": "no (equivale a id)",
    "attributes[].value_id": "no", "attributes[].value_name": "no", "attributes[].attribute_group_id": "no",
    "attributes[].attribute_group_name": "no",
    "pictures[].id": "conteo n_fotos", "pictures[].max_size": "sí: features de fotos (resolución, proporción)",
    "pictures[].size": "no", "pictures[].quality": "no (siempre vacío)", "pictures[].url": "no", "pictures[].secure_url": "no",
}

filas = []
for ruta_texto, como in uso.items():
    base, _, resto = ruta_texto.partition(".")
    base = base.replace("[]", "")
    partes = [p.replace("[]", "") for p in ruta_texto.replace("[]", "").split(".")][1:]
    if "[]" in ruta_texto and not partes:
        partes = []
    if ruta_texto.endswith("]") or ("[]" in ruta_texto and "." not in ruta_texto):
        partes = []
    if "[]." in ruta_texto:
        base = ruta_texto.split("[].")[0]
        partes = ruta_texto.split("[].")[1].split(".")
    valores = valores_campo(crudo[base], partes) if base in crudo.columns else pd.Series([None] * len(crudo))
    filas.append({"campo": ruta_texto, **resumen(valores), "uso_actual": como})
inventario = pd.DataFrame(filas).set_index("campo")
pd.set_option("display.max_rows", 120)
pd.set_option("display.max_colwidth", 70)
inventario

,tipo,pct_nulo_o_vacio,valores_distintos,uso_actual
campo,,,,
id,str,0.0000,70000,no (identificador)
site_id,str,0.0000,1,no (constante)
title,str,0.0000,69334,sí: TF-IDF 500 + modelo de texto (prob_texto_new)
subtitle,—,1.0000,1,no (vacío)
seller_id,int,0.0000,27949,sí: CatBoost crudo; TargetEncoder + conteo/precio (HistGB)
price,float,0.0000,8155,sí: log_price (capeado); bin de precio en combos
base_price,float,0.0000,8155,"solo capeado, no entra al modelo"
currency_id,str,0.0000,2,sí: grupo del capeo; c_currency
initial_quantity,int,0.0000,382,sí


## 2. Representaciones de los campos no usados

Para cada campo no usado (o usado solo como conteo/flag) con variación, una representación simple (`cat` = categórica cruda como texto, `num` = número). Los constantes o vacíos (`site_id`, `subtitle`, `listing_source`, `international_delivery_mode`, `differential_pricing`, `catalog_product_id`, `coverage_areas`, `country`, `pictures[].quality`) no se miden: no tienen información.

In [3]:
def lista_a_texto(lista, clave=None):
    if not isinstance(lista, list) or not lista:
        return ""
    valores = [str(e.get(clave)) if isinstance(e, dict) else str(e) for e in lista] if clave else [str(e) for e in lista]
    return "|".join(sorted(valores))


def numero_mla(texto):
    m = pd.Series(texto).astype(str).str.extract(r"MLA(\d+)")[0]
    return pd.to_numeric(m, errors="coerce").to_numpy()


def numero_descripcion(lista):
    if not isinstance(lista, list) or not lista:
        return np.nan
    try:
        d = ast.literal_eval(lista[0]) if isinstance(lista[0], str) else lista[0]
        return float(str(d.get("id")).split("-")[-1])
    except Exception:
        return np.nan


def suma_variaciones(lista, clave):
    return sum(e.get(clave) or 0 for e in lista) if isinstance(lista, list) and lista else np.nan


def rango_precio_variaciones(lista):
    precios = [e.get("price") for e in lista if isinstance(e, dict) and e.get("price") is not None] if isinstance(lista, list) else []
    return max(precios) - min(precios) if precios else np.nan


inicio = pd.to_datetime(crudo["start_time"])
candidatos = {
    "ciudad": ("cat", crudo["seller_address"].map(lambda d: str(d.get("city", {}).get("name")))),
    "ciudad_id": ("cat", crudo["seller_address"].map(lambda d: str(d.get("city", {}).get("id")))),
    "sub_status": ("cat", crudo["sub_status"].map(lista_a_texto)),
    "deal_ids": ("cat", crudo["deal_ids"].map(lista_a_texto)),
    "base_price_sobre_price": ("num", crudo["base_price"] / crudo["price"]),
    "shipping_methods_nulo": ("num", crudo["shipping"].map(lambda d: int(d.get("methods") is None))),
    "shipping_tags": ("cat", crudo["shipping"].map(lambda d: lista_a_texto(d.get("tags")))),
    "shipping_dimensions": ("cat", crudo["shipping"].map(lambda d: str(d.get("dimensions")))),
    "shipping_free_methods": ("num", crudo["shipping"].map(lambda d: int(d.get("free_methods") is not None))),
    "pagos_tipos": ("cat", crudo["non_mercado_pago_payment_methods"].map(lambda l: lista_a_texto(l, "type"))),
    "variaciones_atributos": ("cat", crudo["variations"].map(
        lambda l: "|".join(sorted({a.get("name", "") for e in l for a in (e.get("attribute_combinations") or [])})) if l else "")),
    "variaciones_custom_field": ("num", crudo["variations"].map(lambda l: int(any(e.get("seller_custom_field") for e in l)) if l else np.nan)),
    "variaciones_n_fotos": ("num", crudo["variations"].map(lambda l: sum(len(e.get("picture_ids") or []) for e in l) if l else np.nan)),
    "variaciones_vendidas": ("num", crudo["variations"].map(lambda l: suma_variaciones(l, "sold_quantity"))),
    "variaciones_disponibles": ("num", crudo["variations"].map(lambda l: suma_variaciones(l, "available_quantity"))),
    "variaciones_rango_precio": ("num", crudo["variations"].map(rango_precio_variaciones)),
    "atributos_valores": ("cat", crudo["attributes"].map(lambda l: lista_a_texto(l, "value_name"))),
    "atributos_grupos": ("cat", crudo["attributes"].map(lambda l: lista_a_texto(l, "attribute_group_id"))),
    "id_numero": ("num", numero_mla(crudo["id"])),
    "parent_id_numero": ("num", numero_mla(crudo["parent_item_id"])),
    "id_menos_parent": ("num", numero_mla(crudo["id"]) - numero_mla(crudo["parent_item_id"])),
    "descripcion_numero": ("num", crudo["descriptions"].map(numero_descripcion)),
    "fotos_size_igual_max": ("num", crudo["pictures"].map(lambda l: np.mean([f.get("size") == f.get("max_size") for f in l]) if l else np.nan)),
    "descuento_original_sobre_price": ("num", crudo["original_price"] / crudo["price"]),
    "dias_creacion_a_inicio": ("num", (inicio - pd.to_datetime(crudo["date_created"]).dt.tz_localize(None)).dt.total_seconds() / 86400),
    "duracion_dias": ("num", (pd.to_datetime(crudo["stop_time"]) - inicio).dt.total_seconds() / 86400),
    "permalink_tipo": ("cat", crudo["permalink"].str.extract(r"https?://([a-z]+)\.")[0].fillna("")),
    "video_id": ("cat", crudo["video_id"].fillna("").astype(str)),
}
for nombre, (tipo, valores) in candidatos.items():
    train[f"x_{nombre}"] = valores.to_numpy() if hasattr(valores, "to_numpy") else valores
print(len(candidatos), "candidatos")
pd.DataFrame({n: {"tipo": t, "distintos": pd.Series(v).astype(str).nunique(), "pct_nulo": round(pd.isna(pd.Series(v)).mean(), 3)}
              for n, (t, v) in candidatos.items()}).T

28 candidatos


,tipo,distintos,pct_nulo
ciudad,cat,3113,0.0
ciudad_id,cat,270,0.0
sub_status,cat,4,0.0
deal_ids,cat,28,0.0
base_price_sobre_price,num,15,0.0
shipping_methods_nulo,num,2,0.0
shipping_tags,cat,3,0.0
shipping_dimensions,cat,10,0.0
shipping_free_methods,num,2,0.0
pagos_tipos,cat,61,0.0


## 3. Señal sola y aporte marginal

- **(a) señal sola:** `HistGB` con solo esa columna (categóricas: `TargetEncoder` → `HistGB`), CV de 5 folds.
- **(b) aporte marginal:** `armar_pipeline_histgb_combos()` (el HistGB P7a de `features.py`, con el modelo de texto cacheado del laboratorio) + la columna (categóricas con `TargetEncoder`, numéricas tal cual). Comparación pareada contra P7a (0,9371).

In [4]:
def p7a_con(columna, tipo):
    pipeline = armar_pipeline_histgb_combos()
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    transformers.append(("nuevo", TargetEncoder(target_type="binary", random_state=SEED) if tipo == "cat" else "passthrough", [columna]))
    return pipeline


def solo(columna, tipo):
    modelo = HistGradientBoostingClassifier(learning_rate=0.1, max_depth=None, max_iter=200, random_state=SEED)
    if tipo == "cat":
        return Pipeline([("te", TargetEncoder(target_type="binary", random_state=SEED)), ("modelo", modelo)])
    return modelo


filas = []
inicio_total = time.time()
for nombre, (tipo, _) in candidatos.items():
    columna = f"x_{nombre}"
    inicio = time.time()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        sola = cross_validate(solo(columna, tipo), train[[columna]], y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
        marginal = cross_validate(p7a_con(columna, tipo), train, y, cv=cv, scoring="accuracy", n_jobs=5)["test_score"]
    diferencias = np.round(marginal - acc_p7a, 10)  # sin el redondeo, diferencias de 1e-17 contaban como "gana"
    gana = int((diferencias > 0).sum())
    filas.append({"campo": nombre, "tipo": tipo, "señal_sola": sola.mean(), "con_P7a": marginal.mean(),
                  "dif_media": diferencias.mean(), "dif_desvio": diferencias.std(), "folds_gana": gana,
                  "pasa": gana == 5 or (gana == 4 and diferencias.mean() > diferencias.std()),
                  "minutos": (time.time() - inicio) / 60})
    print(f"{nombre:32s} sola {sola.mean():.4f} | con P7a {marginal.mean():.4f} | dif {diferencias.mean():+.4f} ({gana}/5) | {(time.time() - inicio) / 60:.1f} min")
print(f"Total: {(time.time() - inicio_total) / 60:.1f} min")
tabla = pd.DataFrame(filas).set_index("campo").sort_values("dif_media", ascending=False)
tabla["señal_sola_alta"] = tabla["señal_sola"] > 0.70
tabla.round(4)

ciudad                           sola 0.6315 | con P7a 0.9364 | dif -0.0007 (1/5) | 0.6 min


ciudad_id                        sola 0.5661 | con P7a 0.9366 | dif -0.0005 (1/5) | 0.6 min


sub_status                       sola 0.5370 | con P7a 0.9362 | dif -0.0009 (1/5) | 0.5 min


deal_ids                         sola 0.5370 | con P7a 0.9366 | dif -0.0005 (1/5) | 0.6 min


base_price_sobre_price           sola 0.5370 | con P7a 0.9371 | dif +0.0000 (4/5) | 0.6 min


shipping_methods_nulo            sola 0.5370 | con P7a 0.9371 | dif -0.0000 (2/5) | 0.6 min


... (19 outputs left out)

duracion_dias                    sola 0.5629 | con P7a 0.9412 | dif +0.0041 (5/5) | 0.5 min


permalink_tipo                   sola 0.5489 | con P7a 0.9366 | dif -0.0005 (1/5) | 0.6 min


video_id                         sola 0.5383 | con P7a 0.9366 | dif -0.0005 (1/5) | 0.5 min
Total: 15.4 min


,tipo,señal_sola,con_P7a,dif_media,dif_desvio,folds_gana,pasa,minutos,señal_sola_alta
campo,,,,,,,,,
duracion_dias,num,0.5629,0.9412,0.0041,0.0008,5,True,0.5167,False
shipping_tags,cat,0.5370,0.9372,0.0001,0.0006,3,False,0.5394,False
variaciones_rango_precio,num,0.5370,0.9371,0.0000,0.0000,4,True,0.5963,False
descuento_original_sobre_price,num,0.5370,0.9371,0.0000,0.0000,4,True,0.5809,False
base_price_sobre_price,num,0.5370,0.9371,0.0000,0.0000,4,True,0.5928,False
shipping_methods_nulo,num,0.5370,0.9371,-0.0000,0.0006,2,False,0.5880,False
fotos_size_igual_max,num,0.6323,0.9370,-0.0001,0.0006,3,False,0.5049,False
shipping_free_methods,num,0.5370,0.9368,-0.0003,0.0005,2,False,0.5318,False
atributos_valores,cat,0.5385,0.9368,-0.0003,0.0011,2,False,0.5492,False


**Lectura del barrido.**
- **Un solo campo aporta: `duracion_dias`** (`stop_time` − `start_time`): +0,0041 sobre el HistGB P7a, **5/5 folds**, aunque solo casi no predice (0,563).
- Cuatro campos figuraban "pasa" con diferencia +0,0000: era un artefacto de redondeo (no cambian ni una predicción). Corregido en la celda anterior: no pasan.
- **Ningún campo tiene señal sola > 0,70.** Los mejores solos son `ciudad` (0,632) y `fotos_size_igual_max` (0,632), que no aportan encima de P7a: lo que dicen ya lo resumen el vendedor y las features de fotos.

### La duración, en detalle

La señal está en los **segundos**, no en los días. Las publicaciones que duran **60 días + unos segundos** (3–117 s, mediana 10) son casi todas `used`.

In [3]:
segundos_extra = (pd.to_datetime(crudo["stop_time"]) - pd.to_datetime(crudo["start_time"])).dt.total_seconds() - 60 * 86400
grupo = pd.Series(np.select([segundos_extra == 0, (segundos_extra > 0) & (segundos_extra <= 120)],
                            ["60 días exactos", "60 días + 1-120 s"], "otra duración"))
resumen_duracion = pd.DataFrame({"filas": grupo.value_counts(), "pct_new": y.groupby(grupo).mean(),
                                 "pct_con_parent_item": crudo["parent_item_id"].notna().groupby(grupo).mean()}).round(3)
dias = (segundos_extra / 86400 + 60).round(4)
detalle = pd.DataFrame({"filas": dias.value_counts().head(10), "pct_new": y.groupby(dias).mean().reindex(dias.value_counts().head(10).index)}).round(3)
print(resumen_duracion)
print()
print(detalle)

                   filas  pct_new  pct_con_parent_item
60 días + 1-120 s   1343    0.025                0.010
60 días exactos    66128    0.552                0.797
otra duración       2529    0.408                0.473

          filas  pct_new
60.0000   66143    0.552
60.0001     807    0.005
7.0000      523    0.321
60.0002     250    0.020
30.0000     184    0.043
60.0003      95    0.063
365.0000     76    0.158
60.0004      40    0.175
60.0005      31    0.000
60.0006      25    0.000


- Casi ninguna tiene `parent_item_id` (1% contra 80% de las de 60 días exactos): son publicaciones creadas por primera vez, no republicadas. Casi todas `bronze`/`silver` y `buy_it_now`.
- **Hipótesis (no verificada):** al crear una publicación desde cero, MercadoLibre calcula el vencimiento unos segundos después del inicio; las republicadas copian 60 días justos. Es un **artefacto de la plataforma**, no del producto.
- **No es leakage:** `stop_time` está en el test con la misma distribución (5,8% de filas no exactas contra 5,5% en train) y se conoce al predecir; no usa el target. Para un sistema real habría que verificar que el campo se genere igual.

### Confirmación en P7c (los dos modelos + promedio 50/50), con las semillas de CV 42 y 7

`duracion_dias` se suma como numérica a CatBoost P6b (las categóricas se corren un lugar: `INDICES_CAT + 1`) y al HistGB P7a; se promedian 50/50 y se compara contra P7c.

In [4]:
from sklearn.model_selection import cross_val_predict
from features import armar_pipeline_final, agregar_categoricas_crudas, COLUMNAS_NUMERICAS, INDICES_CAT

train = agregar_categoricas_crudas(train)


def con_texto_cacheado(pipeline):
    transformers = pipeline.named_steps["preprocesador"].transformers
    i = [n for n, _, _ in transformers].index("texto")
    transformers[i] = ("texto", Pipeline([("oof", PrediccionOOF(armar_modelo_texto(), random_state=SEED)),
                                          ("identidad", "passthrough")], memory=memoria_texto), ["title", "warranty_texto"])
    return pipeline


def catboost_con_duracion():
    pipeline = con_texto_cacheado(armar_pipeline_final())
    transformers = pipeline.named_steps["preprocesador"].transformers
    transformers[0] = ("num", "passthrough", COLUMNAS_NUMERICAS + ["x_duracion_dias"])
    return pipeline, [i + 1 for i in INDICES_CAT]


def histgb_con_duracion():
    pipeline = con_texto_cacheado(armar_pipeline_histgb_combos())
    pipeline.named_steps["preprocesador"].transformers.append(("nuevo", "passthrough", ["x_duracion_dias"]))
    return pipeline


registro = pd.read_csv(archivo_resultados).set_index("nombre")
referencias = {42: "P7c (ref) - CatBoost P6b + HistGB P7a 50/50", 7: "P7c 50/50 (cv semilla 7)"}
resultados_duracion = {}
for semilla in [42, 7]:
    cv_s = StratifiedKFold(n_splits=5, shuffle=True, random_state=semilla)
    inicio = time.time()
    pipe_cat, indices = catboost_con_duracion()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        prob_cat = cross_val_predict(pipe_cat, train, y, cv=cv_s, method="predict_proba", n_jobs=5,
                                     params={"modelo__cat_features": indices})[:, 1]
        prob_hgb = cross_val_predict(histgb_con_duracion(), train, y, cv=cv_s, method="predict_proba", n_jobs=5)[:, 1]
    folds_s = np.empty(len(train), dtype=int)
    for k, (_, v) in enumerate(cv_s.split(train, y)):
        folds_s[v] = k
    def acc(prob):
        return pd.Series((prob >= 0.5).astype(int) == y.to_numpy()).groupby(folds_s).mean().to_numpy()
    acc_prom = acc(0.5 * prob_cat + 0.5 * prob_hgb)
    referencia = registro.loc[referencias[semilla], columnas_folds].to_numpy(dtype=float)
    diferencias = np.round(acc_prom - referencia, 10)
    resultados_duracion[semilla] = {"CatBoost + duración": acc(prob_cat).mean(), "HistGB P7a + duración": acc(prob_hgb).mean(),
                                    "Promedio 50/50 + duración": acc_prom.mean(), "P7c (referencia)": referencia.mean(),
                                    "dif media": diferencias.mean(), "dif desvío": diferencias.std(),
                                    "folds gana": int((diferencias > 0).sum()), "minutos": (time.time() - inicio) / 60}
    print(f"semilla {semilla}: diferencias por fold {diferencias}")
    pd.DataFrame({"fold": np.arange(1, 6), "prob": [None] * 5})
    if semilla == 42:
        pd.DataFrame({"catboost": prob_cat, "histgb": prob_hgb, "fold": folds_s + 1, "y": y.to_numpy()}).to_csv(
            os.path.join(carpeta_lab, "oof_p7c_duracion.csv"), index=False)
pd.DataFrame(resultados_duracion).round(4)

semilla 42: diferencias por fold [0.00478571 0.00392857 0.00521429 0.00328571 0.00478571]


semilla 7: diferencias por fold [0.0055     0.00571429 0.00342857 0.00521429 0.00457143]


,42,7
CatBoost + duración,0.9414,0.9406
HistGB P7a + duración,0.9412,0.9412
Promedio 50/50 + duración,0.9420,0.9422
P7c (referencia),0.9376,0.9373
dif media,0.0044,0.0049
dif desvío,0.0007,0.0008
folds gana,5.0000,5.0000
minutos,2.1057,2.1291


## 4. Conclusión

**Inventario:** 82 campos y sub-campos. Varios no tienen información: constantes (`site_id`, `listing_source`, `international_delivery_mode`, `country`) o vacíos (`subtitle`, `differential_pricing`, `coverage_areas`, `shipping.methods`, `pictures[].quality`). 28 campos no usados o usados solo como conteo/flag se midieron uno por uno.

**Resultado:** **un único campo aporta, y mucho: `duracion_dias`** (`stop_time` − `start_time`).

| | semilla 42 | semilla 7 |
|---|---|---|
| P7c (referencia) | 0,9376 | 0,9373 |
| CatBoost + duración | 0,9414 | 0,9406 |
| HistGB P7a + duración | 0,9412 | 0,9412 |
| **Promedio 50/50 + duración** | **0,9420** (5/5, +0,0044) | **0,9422** (5/5, +0,0049) |

- La señal está en los **segundos**: las publicaciones que duran 60 días + 3–117 segundos (1.343 en train, 2,5% `new`) son casi todas `used`, contra 55% `new` en las de 60 días exactos. Casi ninguna tiene `parent_item_id` (1% contra 80%): son publicaciones creadas por primera vez. Hipótesis: artefacto de cómo la plataforma calcula el vencimiento.
- **Sola casi no predice (0,563)** porque afecta solo al ~2% de las filas. Pero esas filas eran difíciles para el modelo, y ahí acierta casi siempre.
- **No es leakage:** `stop_time` está en el test con la misma distribución y no usa el target.
- **Ningún campo tiene señal sola > 0,70.** Los mejores solos (`ciudad` 0,632, `fotos_size_igual_max` 0,632) no aportan sobre P7a: el vendedor y las features de fotos ya capturan eso.
- Los demás campos (ids, descripciones, variaciones, valores de atributos, envío, pagos, `permalink`, `video_id`, etc.) quedan en el ruido o empeoran.
- Explica buena parte de la distancia con los mejores del leaderboard (~0,945): la CV pasó de 0,9376 a 0,9420.

**Submission (sin subir):** `submission/submission_p7c_duracion.csv` (`19_submission_duracion.ipynb`), 322 predicciones distintas de P7c. **Pendiente:** pasar `duracion_dias` a `features.py` y a `train.ipynb` / `predict.ipynb`.